# 01 — Mobility schema QC and Parquet export

This notebook ingests Google COVID-19 Community Mobility Reports for the United States (2020–2022), performs schema and data quality checks, explores missingness/value distributions, and persists a typed dataset to Parquet for downstream EDA.

Columns (expected 15):
- country_region_code, country_region, sub_region_1, sub_region_2, metro_area, iso_3166_2_code, census_fips_code, place_id, date
- retail_and_recreation_percent_change_from_baseline
- grocery_and_pharmacy_percent_change_from_baseline
- parks_percent_change_from_baseline
- transit_stations_percent_change_from_baseline
- workplaces_percent_change_from_baseline
- residential_percent_change_from_baseline

In [1]:
from hurricane_mobility.config import GOOGLE_MOBILITY_DIR, PROCESSED_DIR, ensure_dirs
from hurricane_mobility.loaders import load_all_google_mobility
from hurricane_mobility.cleaning import optimize_google_mobility
from hurricane_mobility.lookups import EXPECTED_COLUMNS

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

ensure_dirs()

In [2]:
# Load and concatenate 2020-2022
df_all = load_all_google_mobility(GOOGLE_MOBILITY_DIR)
print(df_all.shape)
df_all.head(3)

Loading 2020_US_Region_Mobility_Report.csv
Loading 2021_US_Region_Mobility_Report.csv
Loading 2022_US_Region_Mobility_Report.csv
(2511994, 16)


,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline,year
0,US,United States,<NA>,<NA>,<NA>,<NA>,<NA>,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-15,6,2,15,3,2,-1,2020
1,US,United States,<NA>,<NA>,<NA>,<NA>,<NA>,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-16,7,1,16,2,0,-1,2020
2,US,United States,<NA>,<NA>,<NA>,<NA>,<NA>,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-17,6,0,28,-9,-24,5,2020


In [3]:
# Schema checks: columns, dtypes, memory, basic counts
print('Columns present:', list(df_all.columns))
print('\nDtypes:')
print(df_all.dtypes)

mem_mb = df_all.memory_usage(deep=True).sum() / (1024**2)
print(f"\nApprox memory usage: {mem_mb:.2f} MB")

# Row counts by year
print('\nRow counts by year:')
print(df_all['year'].value_counts(dropna=False).sort_index())

# Geographic level tags
is_national = df_all['sub_region_1'].isna()
is_state = df_all['sub_region_1'].notna() & df_all['sub_region_2'].isna()
is_county = df_all['sub_region_1'].notna() & df_all['sub_region_2'].notna()

level_counts = {
    'national_rows': int(is_national.sum()),
    'state_rows': int(is_state.sum()),
    'county_rows': int(is_county.sum()),
}
print('\nRow counts by geographic level:')
print(level_counts)

# Sanity check: mutually exclusive
assert not (is_national & is_state).any()
assert not (is_state & is_county).any()
assert not (is_national & is_county).any()

Columns present: ['country_region_code', 'country_region', 'sub_region_1', 'sub_region_2', 'metro_area', 'iso_3166_2_code', 'census_fips_code', 'place_id', 'date', 'retail_and_recreation_percent_change_from_baseline', 'grocery_and_pharmacy_percent_change_from_baseline', 'parks_percent_change_from_baseline', 'transit_stations_percent_change_from_baseline', 'workplaces_percent_change_from_baseline', 'residential_percent_change_from_baseline', 'year']

Dtypes:
country_region_code                                   string[python]
country_region                                        string[python]
sub_region_1                                          string[python]
sub_region_2                                          string[python]
metro_area                                            string[python]
iso_3166_2_code                                       string[python]
census_fips_code                                      string[python]
place_id                                              s

In [4]:
# Missingness summary
missing_pct = (df_all.isna().sum() / len(df_all) * 100).sort_values(ascending=False)
missing_pct

metro_area                                            100.000000
iso_3166_2_code                                        98.022527
parks_percent_change_from_baseline                     75.315904
transit_stations_percent_change_from_baseline          61.759582
grocery_and_pharmacy_percent_change_from_baseline      41.934814
residential_percent_change_from_baseline               40.630471
retail_and_recreation_percent_change_from_baseline     35.129901
census_fips_code                                        2.042919
sub_region_2                                            2.016247
workplaces_percent_change_from_baseline                 1.751716
sub_region_1                                            0.038774
country_region_code                                     0.000000
country_region                                          0.000000
place_id                                                0.000000
date                                                    0.000000
year                     

In [5]:
# Value counts for each column (top 20)
for col in df_all.columns:
    if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
        print(f"\n=== value_counts: {col} (top 20) ===")
        print(df_all[col].value_counts(dropna=False).head(20))
    else:
        print(f"\n=== describe: {col} ===")
        print(df_all[col].describe())


=== value_counts: country_region_code (top 20) ===
country_region_code
US    2511994
Name: count, dtype: Int64

=== value_counts: country_region (top 20) ===
country_region
United States    2511994
Name: count, dtype: Int64

=== value_counts: sub_region_1 (top 20) ===
sub_region_1
Texas             185210
Georgia           131364
Virginia          116956
North Carolina     92676
Missouri           90757
Kentucky           90494
Indiana            86391
Illinois           86309
Ohio               84922
Tennessee          84630
Iowa               81035
Michigan           74428
Minnesota          72119
Mississippi        65835
Wisconsin          65298
Florida            64111
Pennsylvania       63420
Alabama            62030
Arkansas           60897
New York           60126
Name: count, dtype: Int64

=== value_counts: sub_region_2 (top 20) ===
sub_region_2
<NA>                 50648
Washington County    26372
Jefferson County     21673
Franklin County      20665
Jackson County       1887

/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
/var/folders/wz/_2x6lz

count    1458594.0
mean      3.827086
std      16.006411
min         -100.0
25%           -5.0
50%            3.0
75%           12.0
max          332.0
Name: grocery_and_pharmacy_percent_change_from_baseline, dtype: Float64

=== describe: parks_percent_change_from_baseline ===
count     620063.0
mean     30.164478
std      63.314871
min          -96.0
25%          -12.0
50%           15.0
75%           58.0
max          812.0
Name: parks_percent_change_from_baseline, dtype: Float64

=== describe: transit_stations_percent_change_from_baseline ===
count     960597.0
mean     -5.027817
std      32.163141
min         -100.0
25%          -27.0
50%           -6.0
75%           12.0
max          478.0
Name: transit_stations_percent_change_from_baseline, dtype: Float64

=== describe: workplaces_percent_change_from_baseline ===
count    2467991.0
mean     -19.29211
std      14.480998
min         -100.0
25%          -27.0
50%          -18.0
75%          -10.0
max          124.0
Name: workplaces_

/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):
/var/folders/wz/_2x6lz2x437d29bql459532c0000gn/T/ipykernel_9904/4116216331.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_string_dtype(df_all[col]) or pd.api.types.is_categorical_dtype(df_all[col]):


In [6]:
# Duplicate checks
dups_place_date = df_all.duplicated(subset=['place_id', 'date'])
print('Duplicate rows (place_id,date):', int(dups_place_date.sum()))

key_cols = ['country_region_code', 'sub_region_1', 'sub_region_2', 'date']
dups_geo_date = df_all.duplicated(subset=key_cols)
print('Duplicate rows (geo fields,date):', int(dups_geo_date.sum()))

if dups_place_date.any():
    print('\nExamples of (place_id,date) duplicates:')
    print(df_all.loc[dups_place_date, ['place_id','date','country_region','sub_region_1','sub_region_2']].head())

if dups_geo_date.any():
    print('\nExamples of (geo fields,date) duplicates:')
    print(df_all.loc[dups_geo_date, key_cols + ['place_id']].head())

Duplicate rows (place_id,date): 0
Duplicate rows (geo fields,date): 0


In [7]:
# Optimize dtypes and persist to Parquet
df_final = optimize_google_mobility(df_all)

out_path = PROCESSED_DIR / 'us_mobility_2020_2022.parquet'
df_final.to_parquet(out_path, index=False)
print(f"Wrote Parquet: {out_path}")
print(f"Shape: {df_final.shape}")
print(f"Memory: {df_final.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Wrote Parquet: /Users/yzc/Desktop/CHIP/hurricane/data/processed/us_mobility_2020_2022.parquet
Shape: (2511994, 16)
Memory: 321.56 MB
